In [1]:
import os
from peft import PeftModel
from unsloth import FastLanguageModel

# =========================
# 0. 配置（必须改）
# =========================
BASE_MODEL = os.path.abspath("./Qwen3-4B-Instruct-2507")  # ← 改成你最初加载底模用的 model_id 或本地路径
OUTPUT_DIR = "./lora_output"     # ← 改成你训练时的 output_dir
CKPT       = os.path.join(OUTPUT_DIR, "checkpoint-200")  # 最近一次 save_steps 存下的适配器
MERGED_DIR = "./merged_model"

# 自动挑选存在的最大 checkpoint（防止 200 没存上，回退到 150/100/50）
def pick_checkpoint(ckpt_dir):
    if os.path.isdir(ckpt_dir) and os.path.exists(
            os.path.join(ckpt_dir, "adapter_model.safetensors")):
        return ckpt_dir
    steps = []
    base = os.path.dirname(ckpt_dir) if False else os.path.dirname(OUTPUT_DIR) or "."
    for name in os.listdir(OUTPUT_DIR):
        if name.startswith("checkpoint-"):
            p = os.path.join(OUTPUT_DIR, name)
            if os.path.exists(os.path.join(p, "adapter_model.safetensors")):
                steps.append((int(name.split("-")[1]), p))
    if not steps:
        raise FileNotFoundError("没找到任何含 adapter_model.safetensors 的 checkpoint")
    best = max(steps)[1]
    print(f"checkpoint-200 不可用，改用 {best}")
    return best

CKPT = pick_checkpoint(CKPT)

# =========================
# 1. 关掉 model card 生成（避开 'CardData' 报错）
# =========================
PeftModel.create_or_update_model_card = lambda self, output_dir: None

# =========================
# 2. 重新加载底模（与训练时同一套加载逻辑）
# =========================
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    load_in_4bit=False,   # 合并建议用非量化底模；显存不够可设 True
)

# 3. 挂上 checkpoint 里的 LoRA 适配器
model = PeftModel.from_pretrained(base_model, CKPT)

# =========================
# 4. 保存为干净的 LoRA 适配器目录
# =========================
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("LoRA 已保存:",
      os.path.exists(os.path.join(OUTPUT_DIR, "adapter_model.safetensors"))
      or os.path.exists(os.path.join(OUTPUT_DIR, "adapter_model.safetensors".replace("sensors","sensors"))))

# =========================
# 5. 合并成完整模型
# =========================
# 合并前确保不在推理包装态
try:
    model = FastLanguageModel.for_inference(model, enabled=False)
except Exception:
    pass

merged_model = model.merge_and_unload()
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print("合并模型已保存:", os.path.exists(MERGED_DIR))

W1015 17:44:55.029000 5060 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1015 17:44:55.060000 5060 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
/usr/local/lib/python3.12/site-packages/unsloth/__init__.py:1633: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


[xformers|WARNING]WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.10.0+cu128 with CUDA 1208 (you have 2.13.0+cu130)
    Python  3.10.19 (you have 3.12.13)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details
[transformers] `Siglip2ImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Siglip2ImageProcessor` instead.


Unsloth: xformers is installed but cannot load with torch 2.13.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.5: Fast Qwen3 patching. Transformers: 5.19.0. vLLM: 0.27.1.
   \\   /|    NVIDIA A10. Num GPUs = 1. Max memory: 22.184 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.13.0+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
==((====))==  Unsloth 2026.10.5: Fast Qwen3 patching. Transformers: 5.19.0. vLLM: 0.27.1.
   \\   /|    NVIDIA A10. Num GPUs = 1. Max memory: 22.184 GB. Platform: Linux

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Unsloth: Restored added_tokens_decoder metadata in ./lora_output/tokenizer_config.json.


LoRA 已保存: True


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Unsloth: Restored added_tokens_decoder metadata in ./merged_model/tokenizer_config.json.


合并模型已保存: True
